In [ ]:
# Lot 2d étape 3 (CLASSE B) : recollage, contrôle des sha, exécution de reader_gate_calib_2d.py
import glob, hashlib, os, subprocess, sys, tarfile, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz'], check=True)
parts = sorted(glob.glob('/kaggle/input/**/part_aa?', recursive=True), key=os.path.basename)
assert len(parts) == 5, parts
A = '/kaggle/working/inputs.tar.gz'
with open(A, 'wb') as g:
    for q in parts:
        g.write(open(q, 'rb').read())
assert hashlib.sha256(open(A, 'rb').read()).hexdigest() == 'c112e085a2dd9902770d6e245cb87f28aec205d450d1b0b5c8fadfccdf148315'
H = '/kaggle/working/home'; os.makedirs(H, exist_ok=True)
tarfile.open(A).extractall(H)
os.remove(A)
SUMS = """5e272cfbb05dd02013ef739cede90b0f1a522e432717f855f524ac888525f672  axel/runs/e2/lm.press8x.pkl
9cc562fecfe015aadf87243329233615da9c87f023751745e7fe98525521155f  axel/runs/e17/kaggle/a1/timing.json
f800c469bab71ca98bd33c3ddb7e62f062ec439f7d08decd89ccf8d34fbf6163  mathematicien/proto/q3_weights.py
e3484f81f363f573aa65caa51c3349cdbfca58466da6f833ef0918ed99b73f60  babaorum/scripts/reader_gate_calib_2d.py
ba2e5e07112ccced4af0f84d133075c6904b1d48c0633cf91a655937d6ad2ec9  babaorum/scripts/reader_r1d_joint.py
e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855  babaorum/src/babaorum/__init__.py
de7a69c99d4fb4accdfe9686478fc7704e8f72bc591b55706258001d4b230926  babaorum/src/babaorum/reader/__init__.py
9c3da9080181dd4e9811786389482940e513f094fdaa6624c84e84e3eb857de2  babaorum/src/babaorum/reader/ancre.py
e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855  axel/src/axel/__init__.py
3100cfa106e63bf271852e6b9503a177a62b28751d1ef7f706ccfa502b91b26c  axel/src/axel/image/__init__.py
d28150ed3ad06601db886d7ec178a7569c2027fefb20522e08969ff4de7439ef  axel/src/axel/image/fusion.py
313d6e7d62a6a9da57c4dc41827ff81369fdd45b68875f0afb1cff798f0e37d9  axel/src/axel/image/lattice.py
05f8b46dffcdd8894354aeb144a3a411cb8bc4027a4d10d696f4c23fa62346cd  axel/src/axel/reader/__init__.py
fe7bc06b1e330fba54500a3452741b7aedbbbe490b8653203298d6cd8b361a86  axel/src/axel/reader/metric.py
4d98abef76fba132e48d972d83a472c0854b8201bca17454a7283094d1e9b6fe  axel/src/axel/systems/__init__.py
7e8500a874c0bdac63ea055ed59388f0fe569f1bb43c1eaec8ec71d037e610d6  axel/src/axel/systems/charlm.py
8135f2375cf53f060235fe8ca0335708257086145cbe73cdec5f23c1cf62a540  babaorum/reports/reader_items_photo1983.jsonl
58227a07f847eac5304c85456c7146f2794888766bb5589d8dfa9cf69f39b5ba  babaorum/reports/reader_a1_photo1983_blank.jsonl.gz
a03a8b4146b689697f84c5ade033fee34bb4ef36bb60ec352cd47b9c880a10a2  babaorum/reports/reader_a1_photo1983_blur_strong.jsonl.gz
e20608492b3e3d9c5d4289cff3eb028bbca4e50119aa05ec15d5867012d5508a  babaorum/reports/reader_a1_photo1983_noise_strong.jsonl.gz
94ac775745478eedb7d4ad5fb293b730df209dec1027211218fd077b7a5aceef  babaorum/reports/reader_a1_photo1983_scramble.jsonl.gz
305f9b01f4e1d7b22fd21f9b533d33cd565edbae080d6d984a682f0dba84be47  babaorum/reports/reader_a1_photo1983.jsonl.gz
9ebf48ab6f01bd77b246328a67977766a1ad1969f4fe8d4bf09e8801a0765de8  babaorum/reports/reader_kraken_photo1983_blank.jsonl.gz
dc7ed59dfb8446fd53de993f5f10721c3ca70d716cb6926bc8618f1200b5df1a  babaorum/reports/reader_kraken_photo1983_blur_strong.jsonl.gz
75f4988c747ed1eb48006c01500380b498a2b433d47ff795bea0bed653af1d39  babaorum/reports/reader_kraken_photo1983_noise_strong.jsonl.gz
fdf31fe059b9223aa8cea28be90ca6aa565a607b91cb49cdc564d759c46e2728  babaorum/reports/reader_kraken_photo1983_scramble.jsonl.gz
b69c80fb33122c2ea63729d01fa9f016ad8b4c50add1ef0175f6b02d78235ac3  babaorum/reports/reader_kraken_photo1983.jsonl.gz"""
bad = []
for l in SUMS.splitlines():
    h, p = l.split(None, 1)
    if hashlib.sha256(open(os.path.join(H, p.strip()), 'rb').read()).hexdigest() != h:
        bad.append(p)
print('sha256 contrôlés :', len(SUMS.splitlines()), 'différents :', bad)
assert not bad
r = subprocess.run([sys.executable, 'babaorum/scripts/reader_gate_calib_2d.py'], cwd=H, env=dict(os.environ, HOME=H, PYTHONPATH=H + '/axel/src'))
print('rc =', r.returncode)
assert r.returncode == 0, 'reader_gate_calib_2d.py a échoué'
for f in ('reader_gate_calib_2d.json', 'reader_gate_calib_2d_rows.json'):
    shutil.copy(os.path.join(H, 'babaorum/reports', f), '/kaggle/working/' + f)
shutil.rmtree(H)
print(open('/kaggle/working/reader_gate_calib_2d.json').read()[:3000])
